In [1]:
import sys
import warnings
from pathlib import Path
import random

import numpy as np
import pandas as pd
import geopandas as gpd
import scipy.sparse as sp
from scipy.sparse.linalg import lgmres, spsolve
import cv2
import rasterio
from rasterio.features import rasterize as rio_rasterize
from rasterio.transform import rowcol
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


sys.path.insert(0, str(Path.cwd().parent))
import shared



In [2]:
# Variables and paths
figures_dir = shared.OUT / "figures" / "circuit"
figures_dir.mkdir(parents=True, exist_ok=True)

vector_gis_dir = shared.OUT / "vector_gis"
vector_gis_dir.mkdir(exist_ok=True)

footprints, doors_polygons, doors_points, crosswalk = shared.get_base_preprocessed_data()
dem = shared.load_dem()
doors_points = shared.sample_dem_at_doors(doors_points)

hillshade = shared.hillshade(dem["disp"])
xmin, ymin, xmax, ymax = footprints.total_bounds

Loading DEM ...
  CRS=EPSG:32636  shape=(5138, 3902)  res=0.40 m  elev=65.2-145.0 m
Door elevation stats:
count    217.000000
mean     105.879623
std        9.303591
min       87.141602
25%       99.331055
50%      105.127930
75%      112.576172
max      125.653320


In [ ]:
# Computation functions
def get_offsets(): 
    return [(0, 1, 1.0), (1, 0, 1.0), (1, 1, 2**0.5), (1, -1, 2**0.5)]






In [ ]:
def build_conductance_matrix(cost_array, connectivity=8):
    height, width = cost_array.shape
    num_pixels = height * width
    offsets = get_offsets() if connectivity == 8 else [(0, 1, 1.0), (1, 0, 1.0)]
    rows, cols, conductances = [], [], []
    for d_row, d_col, dist in offsets:
        col_start = max(0, -d_col)
        col_end = min(width, width - d_col)
        if d_row >= height or col_start >= col_end: 
            continue
        r, c = np.meshgrid(np.arange(0, height - d_row), np.arange(col_start, col_end), indexing='ij')
        r, c = r.ravel(), c.ravel()
        target_r, target_c = r + d_row, c + d_col
        idx1, idx2 = r * width + c, target_r * width + target_c
        cost1, cost2 = cost_array[r, c], cost_array[target_r, target_c]
        avg_cost = np.maximum(0.5 * (cost1 + cost2), 1e-6)
        conductance = 1.0 / (avg_cost * dist)
        conductance[(cost1 >= 1e8) | (cost2 >= 1e8)] = 0.0
        rows += [idx1, idx2]
        cols += [idx2, idx1]
        conductances += [conductance, conductance]
    
    if rows:
        rows = np.concatenate(rows)
        cols = np.concatenate(cols)
        conductances = np.concatenate(conductances)
        valid_mask = conductances > 1e-12
        rows, cols, conductances = rows[valid_mask], cols[valid_mask], conductances[valid_mask]
    else:
        rows = cols = conductances = np.array([], dtype=np.float64)
        
    diagonal = np.bincount(rows.astype(int), weights=conductances, minlength=num_pixels)
    diagonal[diagonal < 1e-12] = 1.0
    all_rows = np.concatenate([rows, np.arange(num_pixels)])
    all_cols = np.concatenate([cols, np.arange(num_pixels)])
    all_values = np.concatenate([-conductances, diagonal])
    
    return sp.coo_matrix((all_values, (all_rows, all_cols)), shape=(num_pixels, num_pixels)).tocsr()

In [ ]:
def snap_pixels(pixels, mask_array):
    height, width = mask_array.shape
    snapped_pixels = []
    for r, c in pixels:
        if mask_array[r, c] == 1:
            snapped = False
            for radius in range(1, 15):
                for d_row in range(-radius, radius + 1):
                    for d_col in range(-radius, radius + 1):
                        if abs(d_row) != radius and abs(d_col) != radius: 
                            continue
                        new_r, new_c = r + d_row, c + d_col
                        if 0 <= new_r < height and 0 <= new_c < width and mask_array[new_r, new_c] == 0:
                            r, c = new_r, new_c
                            snapped = True
                            break
                    if snapped: 
                        break
                if snapped: 
                    break
        snapped_pixels.append((r, c))
    return snapped_pixels

In [ ]:

def solve_linear_system(laplacian, current_vector):
    try:
        voltage, info = lgmres(laplacian, current_vector, tol=1e-5, maxiter=300)
        if info != 0: 
            raise RuntimeError
    except Exception:
        voltage = spsolve(laplacian, current_vector)
    return voltage

def compute_current_density(voltage, cost_array, connectivity=8):
    height, width = cost_array.shape
    num_pixels = height * width
    current_density_array = np.zeros(num_pixels)
    offsets = get_offsets() if connectivity == 8 else [(0, 1, 1.0), (1, 0, 1.0)]
    for d_row, d_col, dist in offsets:
        col_start = max(0, -d_col)
        col_end = min(width, width - d_col)
        if d_row >= height or col_start >= col_end: 
            continue
        r, c = np.meshgrid(np.arange(0, height - d_row), np.arange(col_start, col_end), indexing='ij')
        r, c = r.ravel(), c.ravel()
        target_r, target_c = r + d_row, c + d_col
        idx1, idx2 = r * width + c, target_r * width + target_c
        cost1, cost2 = cost_array[r, c], cost_array[target_r, target_c]
        avg_cost = np.maximum(0.5 * (cost1 + cost2), 1e-6)
        conductance = 1.0 / (avg_cost * dist)
        conductance[(cost1 >= 1e8) | (cost2 >= 1e8)] = 0.0
        abs_current = np.abs(conductance * (voltage[idx1] - voltage[idx2])) * 0.5
        np.add.at(current_density_array, idx1, abs_current)
        np.add.at(current_density_array, idx2, abs_current)
    return current_density_array.reshape((height, width))

def downsample_cost(cost_array, pixels, max_pixels=250_000):
    height, width = cost_array.shape
    if height * width > max_pixels:
        scale = (max_pixels / (height * width)) ** 0.5
        down_height = max(1, int(round(height * scale)))
        down_width = max(1, int(round(width * scale)))
        down_cost = cv2.resize(cost_array, (down_width, down_height), interpolation=cv2.INTER_NEAREST)
        down_pixels = [
            (int(np.clip(round(r * down_height / height), 0, down_height - 1)),
             int(np.clip(round(c * down_width / width), 0, down_width - 1))) 
            for r, c in pixels
        ]
    else:
        down_height, down_width, down_cost, down_pixels = height, width, cost_array.copy(), list(pixels)
    obstacle_mask = (down_cost > 1e6).astype(np.uint8)
    return down_cost, snap_pixels(down_pixels, obstacle_mask), down_height, down_width, height, width

def apply_dirichlet_boundary(laplacian, sink_index):
    laplacian_copy = laplacian.copy()
    start_ptr = laplacian_copy.indptr[sink_index]
    end_ptr = laplacian_copy.indptr[sink_index + 1]
    laplacian_copy.data[start_ptr:end_ptr] = 0.0
    diag_indices = np.where(laplacian_copy.indices[start_ptr:end_ptr] == sink_index)[0]
    if len(diag_indices): 
        laplacian_copy.data[start_ptr + diag_indices[0]] = 1.0
    return laplacian_copy

def compute_supernode_current(cost_array, pixels, shape, sink_index=None):
    down_cost, down_pixels, down_height, down_width, height, width = downsample_cost(cost_array, pixels)
    if sink_index is None:
        coords = np.array(pixels)
        sink_index = int(np.argmin(np.hypot(coords[:, 0] - coords[:, 0].mean(), coords[:, 1] - coords[:, 1].mean())))
    
    laplacian = build_conductance_matrix(down_cost)
    num_pixels = down_height * down_width
    current_vector = np.zeros(num_pixels)
    sink_flat = down_pixels[sink_index][0] * down_width + down_pixels[sink_index][1]
    
    for i, (r, c) in enumerate(down_pixels):
        flat_idx = r * down_width + c
        if flat_idx != sink_flat: 
            current_vector[flat_idx] += 1.0
            
    current_vector[sink_flat] = -current_vector.sum()
    laplacian_constrained = apply_dirichlet_boundary(laplacian, sink_flat)
    current_vector_constrained = current_vector.copy()
    current_vector_constrained[sink_flat] = 0.0
    
    voltage = solve_linear_system(laplacian_constrained, current_vector_constrained)
    density = cv2.resize(compute_current_density(voltage, down_cost), (shape[1], shape[0]), interpolation=cv2.INTER_LINEAR)
    
    if density.max() > 0: 
        density /= density.max()
    return density

def compute_pairwise_current(cost_array, pixels, max_pairs=50):
    down_cost, down_pixels, down_height, down_width, height, width = downsample_cost(cost_array, pixels)
    pairs = [(down_pixels[i], down_pixels[j]) for i in range(len(down_pixels)) for j in range(i + 1, len(down_pixels)) if down_pixels[i] != down_pixels[j]]
    
    if len(pairs) > max_pairs:
        random.seed(42)
        pairs = random.sample(pairs, max_pairs)
        
    laplacian = build_conductance_matrix(down_cost)
    accumulated_density = np.zeros((down_height, down_width))
    num_pixels = down_height * down_width
    
    for i, (p1, p2) in enumerate(pairs):
        current_vector = np.zeros(num_pixels)
        flat1, flat2 = p1[0] * down_width + p1[1], p2[0] * down_width + p2[1]
        current_vector[flat1] = 1.0
        current_vector[flat2] = -1.0
        
        laplacian_constrained = apply_dirichlet_boundary(laplacian, flat2)
        current_vector_constrained = current_vector.copy()
        current_vector_constrained[flat2] = 0.0
        
        voltage = solve_linear_system(laplacian_constrained, current_vector_constrained)
        accumulated_density += compute_current_density(voltage, down_cost)
        
    density = cv2.resize(accumulated_density, (width, height), interpolation=cv2.INTER_LINEAR)
    
    if density.max() > 0: 
        density /= density.max()
    return density

In [ ]:
# Compute Tobler cost surface
footprints_dem = footprints.to_crs(dem["crs"]) if footprints.crs != dem["crs"] else footprints.copy()
grad_y, grad_x = np.gradient(dem["disp"], dem["res"], dem["res"])
tobler_cost = 1.0 / np.maximum(6.0 * np.exp(-3.5 * np.abs(np.sqrt(grad_x**2 + grad_y**2) + 0.05)) / 3.6, 1e-6)

building_mask = rio_rasterize(
    [(geom, 1) for geom in footprints_dem.geometry if geom is not None],
    out_shape=dem["arr"].shape, transform=dem["transform"], fill=0, dtype=np.uint8
)
cost_surface = 0.5 * tobler_cost + np.where(building_mask == 1, 1e9, 0.0)

profile = dem["profile"].copy()
profile.update(dtype="float32", count=1)
with rasterio.open(shared.OUT / "cost_surface_tobler.tif", "w", **profile) as dst:
    dst.write(cost_surface.astype(np.float32), 1)

# Project doors to raster pixels
rows_px, cols_px = rowcol(dem["transform"], [geom.x for geom in doors_points.geometry], [geom.y for geom in doors_points.geometry])
height, width = dem["arr"].shape
entrance_pixels, valid_doors = [], []

for i, (r, c) in enumerate(zip(rows_px, cols_px)):
    if not (0 <= r < height and 0 <= c < width): 
        continue
    if building_mask[r, c] == 1:
        snapped = False
        for d_row in range(-5, 6):
            for d_col in range(-5, 6):
                new_r, new_c = r + d_row, c + d_col
                if 0 <= new_r < height and 0 <= new_c < width and building_mask[new_r, new_c] == 0:
                    r, c = new_r, new_c
                    snapped = True
                    break
            if snapped: 
                break
    entrance_pixels.append((r, c))
    valid_doors.append(doors_points.iloc[i])

sink_idx = next((i for i, row in enumerate(valid_doors) if str(row["chapel_id"]) == "180"), None)

# Solve supernode current density
supernode_density = compute_supernode_current(cost_surface, entrance_pixels, cost_surface.shape, sink_index=sink_idx)
with rasterio.open(shared.OUT / "circuit_current_supernode.tif", "w", **profile) as dst:
    dst.write(supernode_density.astype(np.float32), 1)

# Solve pairwise current density
pairwise_density = compute_pairwise_current(cost_surface, entrance_pixels, max_pairs=50)
with rasterio.open(shared.OUT / "circuit_current_pairwise.tif", "w", **profile) as dst:
    dst.write(pairwise_density.astype(np.float32), 1)

# Data export
doors_points.to_crs("EPSG:4326").to_file(str(vector_gis_dir / "circuit_door_points.geojson"), driver="GeoJSON")

In [ ]:
# Graph and Plotting
def plot_density(density, title, filename):
    fig, ax = plt.subplots(figsize=(18, 15))
    ax.imshow(dem["disp"], extent=dem["extent"], origin="upper", cmap="terrain", alpha=0.55, vmin=dem["e_min"], vmax=dem["e_max"])
    ax.imshow(hillshade, extent=dem["extent"], origin="upper", cmap="gray", alpha=0.25)
    
    masked_density = np.where(density > 0.01, density, np.nan)
    image = ax.imshow(masked_density, extent=dem["extent"], origin="upper", cmap="hot_r", alpha=0.75, vmin=0, vmax=1)
    
    for _, row in doors_polygons.iterrows():
        x_coords, y_coords = row.geometry.xy
        ax.plot(x_coords, y_coords, color=shared.DIR_CLR.get(row["direction"], "#aaa"), linewidth=1.2, zorder=5)
        
    plt.colorbar(image, ax=ax, label="Current Density (normalised)", shrink=0.5)
    ax.set_title(title)
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.set_xlim(xmin - 50, xmax + 50)
    ax.set_ylim(ymin - 50, ymax + 50)
    
    plt.tight_layout()
    shared.save_fig(figures_dir, filename)
    plt.show()

In [ ]:
plot_density(supernode_density, "Circuit Model — Supernode Current Density (all → Chapel 180)", "17_circuit_supernode_density.png")

In [ ]:
plot_density(pairwise_density, "Circuit Model — Pairwise Current Density (max_pairs=50)", "18_circuit_pairwise_density.png")